In [1]:
!pip install -q transformers einops accelerate langchain bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 10.3 MB/s eta 0:00:00


In [2]:
!nvidia-smi

Wed Jan  1 08:56:03 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              11W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
pip install langchain

In [6]:
pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.25
    Uninstalling langchain-core-0.3.25:
      Successfully uninstalled langchain-core-0.3.25
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.12
    Uninstalling langchain-0.3.12:
      Successfully uninstalled langchain-0.3.12


In [7]:
from langchain import HuggingFacePipeline
from transformers import AutoTokenizer, pipeline
import torch

## so this was the pre-trained I have to see how does this goes some tells that it is better than LLama model, but let's see.
model = "tiiuae/falcon-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model)

pipeline = pipeline(
    "text-generation", #task
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
    max_length=200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id
)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Device set to use cuda:0


In [8]:
llm = HuggingFacePipeline(pipeline = pipeline, model_kwargs = {'temperature':0})

<ipython-input-8-9410ce7de563>:1: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline = pipeline, model_kwargs = {'temperature':0})


In [10]:
from langchain import PromptTemplate,  LLMChain

template = """
You are an Falcon AI Assistant . Help the following question with suitable answers.
Question: {question}
Answer:"""
prompt = PromptTemplate(template=template, input_variables=["question"])


In [11]:
llm_chain  = LLMChain(prompt=prompt, llm=llm)
question = "What is the uses of Machine Learning and what is the benefits of AI in technology??"
print(llm_chain.run(question))

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.



You are an Falcon AI Assistant . Help the following question with suitable answers.
Question: What is the uses of Machine Learning and what is the benefits of AI in technology??
Answer:

The uses of Machine Learning involve analyzing massive amounts of data to extract patterns and insights from them. This can be done using supervised and unsupervised learning algorithms to develop predictive models or to detect fraud and anomalies in data.

Benefits of AI in technology include:

1. Increased Efficiency
2. Faster Decision Making
3. Automation in Daily Tasks
4. Accuracy and Precision in Data Analysis
5. Personalization in User Experience

These benefits of AI are being increasingly utilized in a number of industries, from healthcare to retail to finance.

Overall, the combination of Machine Learning and AI can be used to solve a variety of complex tasks and can have a profound effect on businesses.


The uses of Machine Learning involve analyzing massive amounts of data to extract patterns and insights from them. This can be done using supervised and unsupervised learning algorithms to develop predictive models or to detect fraud and anomalies in data.

Benefits of AI in technology include:

1. Increased Efficiency
2. Faster Decision Making
3. Automation in Daily Tasks
4. Accuracy and Precision in Data Analysis
5. Personalization in User Experience

These benefits of AI are being increasingly utilized in a number of industries, from healthcare to retail to finance.

Overall, the combination of Machine Learning and AI can be used to solve a variety of complex tasks and can have a profound effect on businesses.